# Airbnb Hotel Booking Analysis
**Dataset:** New York City Airbnb Open Data  |  **Program:** VOIS for Tech / Edunet Foundation

**Goal:** Uncover booking, pricing, guest-preference and host-performance patterns to support data-driven decisions for hosts and the platform.

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
PALETTE = ['#B85042', '#E7A11A', '#A7BEAE', '#6D2E46', '#50808E']

In [ ]:
df = pd.read_excel('Airbnb_Open_Data.xlsx')   # or pd.read_csv('Airbnb_Open_Data.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum()

## 2. Data Cleaning

Steps applied:
1. Drop duplicate records
2. Drop `house_rules` and `license` (insufficient data)
3. Remove `$` from `price` / `service fee`
4. Remove commas from `price` / `service fee`
5. Rename those columns to include `($)`
6. Drop records with missing values
7. Fix mismatched data types
8. Correct spelling `brooklyn` -> `Brooklyn`
9. Remove outliers in `availability 365`

In [ ]:
# 1. Duplicates
before = len(df)
df = df.drop_duplicates()
print('Duplicates removed:', before - len(df))

# 2. Drop sparse columns
df = df.drop(columns=['house_rules', 'license'])

# 3 & 4. Strip $ and , then convert to numeric
for c in ['price', 'service fee']:
    df[c] = pd.to_numeric(
        df[c].astype(str).str.replace('$', '', regex=False)
                         .str.replace(',', '', regex=False).str.strip(),
        errors='coerce')

# 5. Rename with currency unit
df = df.rename(columns={'price': 'price ($)', 'service fee': 'service fee ($)'})

# 8. Fix spelling
df['neighbourhood group'] = df['neighbourhood group'].replace(
    {'brookln': 'Brooklyn', 'brooklyn': 'Brooklyn', 'manhatan': 'Manhattan'})

# 6. Drop missing values
df = df.dropna()

# 7. Fix data types
int_cols = ['Construction year', 'minimum nights', 'number of reviews',
            'review rate number', 'calculated host listings count', 'availability 365']
df[int_cols] = df[int_cols].astype(int)

# 9. Outliers in availability 365 (valid range 0-365)
df = df[(df['availability 365'] >= 0) & (df['availability 365'] <= 365)]

print('Clean shape:', df.shape)
df.describe()

## 3. Exploratory Data Analysis

### Q1. What are the different property (room) types in the dataset?

In [ ]:
room_counts = df['room type'].value_counts()
print(room_counts)

plt.figure(figsize=(7, 4))
sns.barplot(x=room_counts.values, y=room_counts.index, palette=PALETTE[:len(room_counts)],
            hue=room_counts.index, legend=False)
plt.title('Listings by Room Type'); plt.xlabel('Listings'); plt.tight_layout(); plt.show()

**Insight:** Four property types exist — Entire home/apt (~42.8k) dominates, followed by Private room (~36.4k); Shared room and Hotel room are marginal.

### Q2. Which neighbourhood group has the highest number of listings?

In [ ]:
ng = df['neighbourhood group'].value_counts()
print(ng)

plt.figure(figsize=(7, 4))
sns.barplot(x=ng.index, y=ng.values, palette=PALETTE, hue=ng.index, legend=False)
plt.title('Listings by Neighbourhood Group'); plt.ylabel('Listings')
plt.xticks(rotation=20); plt.tight_layout(); plt.show()

**Insight:** Brooklyn leads (~33.6k), narrowly ahead of Manhattan (~33.4k). Together they hold over 80% of all NYC listings.

### Q3. Which neighbourhood groups have the highest average prices?

In [ ]:
avg_price = df.groupby('neighbourhood group')['price ($)'].mean().sort_values(ascending=False)
print(avg_price.round(2))

plt.figure(figsize=(7, 4))
sns.barplot(x=avg_price.index, y=avg_price.values, palette=PALETTE, hue=avg_price.index, legend=False)
plt.title('Average Price by Neighbourhood Group ($)'); plt.ylabel('Average price ($)')
plt.xticks(rotation=20); plt.tight_layout(); plt.show()

**Insight:** Average prices are remarkably flat (~$623-$631). The Bronx tops the list marginally — price in this dataset is not driven by borough.

### Q4. Is there a relationship between construction year and price?

In [ ]:
corr_year = df['Construction year'].corr(df['price ($)'])
print('Correlation (construction year vs price):', round(corr_year, 4))

yearly = df.groupby('Construction year')['price ($)'].mean()
plt.figure(figsize=(7, 4))
plt.plot(yearly.index, yearly.values, marker='o', color=PALETTE[0])
plt.title('Average Price by Construction Year'); plt.xlabel('Construction year')
plt.ylabel('Average price ($)'); plt.tight_layout(); plt.show()

**Insight:** Correlation is ~-0.005 — effectively zero. Construction year has no meaningful influence on price.

### Q5. Top 10 hosts by calculated host listings count

In [ ]:
top_hosts = df.groupby('host name')['calculated host listings count'].max().sort_values(ascending=False).head(10)
print(top_hosts)

plt.figure(figsize=(7, 4))
sns.barplot(x=top_hosts.values, y=top_hosts.index, palette=PALETTE * 2,
            hue=top_hosts.index, legend=False)
plt.title('Top 10 Hosts by Listing Count'); plt.xlabel('Listings'); plt.tight_layout(); plt.show()

**Insight:** Professional operators dominate — Blueground (332) and Sonder NYC (327) far outpace individual hosts.

### Q6. Are hosts with verified identities more likely to receive positive reviews?

In [ ]:
verified = df.groupby('host_identity_verified').agg(
    avg_review_rate=('review rate number', 'mean'),
    avg_number_of_reviews=('number of reviews', 'mean'),
    listings=('id', 'count'))
print(verified.round(3))

plt.figure(figsize=(6, 4))
sns.barplot(x=verified.index, y=verified['avg_review_rate'], palette=PALETTE,
            hue=verified.index, legend=False)
plt.ylim(0, 5); plt.title('Average Review Rate by Host Verification')
plt.ylabel('Avg review rate (stars)'); plt.tight_layout(); plt.show()

**Insight:** Verified 3.291 vs unconfirmed 3.284 stars — a negligible difference. Verification alone does not drive better ratings.

### Q7. Correlation between price and service fee

In [ ]:
corr_fee = df['price ($)'].corr(df['service fee ($)'])
print('Correlation (price vs service fee):', round(corr_fee, 4))

sample = df.sample(4000, random_state=1)
plt.figure(figsize=(6, 4))
plt.scatter(sample['price ($)'], sample['service fee ($)'], s=6, alpha=0.4, color=PALETTE[0])
plt.title(f'Price vs Service Fee (r = {corr_fee:.2f})')
plt.xlabel('Price ($)'); plt.ylabel('Service fee ($)'); plt.tight_layout(); plt.show()

**Insight:** Perfect correlation (r = 1.00). Service fee is exactly 20% of price — a deterministic platform rule, not an independent variable.

### Q8. Average review rate, by neighbourhood group and room type

In [ ]:
print('Overall average review rate:', round(df['review rate number'].mean(), 3))

pivot = df.pivot_table(index='neighbourhood group', columns='room type',
                       values='review rate number', aggfunc='mean').round(2)
display(pivot)

plt.figure(figsize=(7, 4))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrBr')
plt.title('Average Review Rate by Group & Room Type'); plt.tight_layout(); plt.show()

**Insight:** Overall average is 3.29 stars with very little variation (3.25-3.83). Hotel rooms in Brooklyn score highest (3.83); Staten Island shared/private rooms also over-index.

### Q9. Do hosts with more listings maintain higher availability?

In [ ]:
bins = [0, 1, 5, 20, 100, 1000]
labels = ['1', '2-5', '6-20', '21-100', '100+']
df['host_size'] = pd.cut(df['calculated host listings count'], bins=bins, labels=labels, include_lowest=True)
avail = df.groupby('host_size', observed=True)['availability 365'].mean()
print(avail.round(1))
print('Correlation:', round(df['calculated host listings count'].corr(df['availability 365']), 4))

plt.figure(figsize=(7, 4))
sns.barplot(x=avail.index.astype(str), y=avail.values, palette=PALETTE,
            hue=avail.index.astype(str), legend=False)
plt.title('Average Availability (of 365 days) by Host Listing Count')
plt.xlabel('Listings owned by host'); plt.ylabel('Average days available')
plt.tight_layout(); plt.show()

**Insight:** Yes. Availability rises from 108 days (single-listing hosts) to 243 days (21-100 listings); correlation +0.15. Professional hosts keep calendars open far longer.

### Correlation Matrix (all numeric features)

In [ ]:
num = df[['price ($)', 'service fee ($)', 'minimum nights', 'number of reviews',
          'review rate number', 'calculated host listings count', 'availability 365',
          'Construction year']]
plt.figure(figsize=(7, 5))
sns.heatmap(num.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix'); plt.tight_layout(); plt.show()

## 4. Conclusions and Recommendations

**Key findings**
1. Four property types; Entire home/apt and Private room make up 98% of supply.
2. Brooklyn has the most listings, with Manhattan a close second (80%+ of the market combined).
3. Average price is essentially uniform across boroughs (~$623-$631) — location does not explain price in this dataset.
4. Construction year has no relationship to price (r = -0.005).
5. Listing supply is concentrated in professional operators (Blueground, Sonder NYC).
6. Identity verification has no measurable effect on ratings (3.29 vs 3.28).
7. Service fee is a fixed 20% of price (r = 1.00) — redundant for modelling.
8. Average rating is 3.29 stars, stable across boroughs and room types.
9. Multi-listing hosts keep far higher annual availability (108 -> 243 days).

**Recommendations**
- Price on room type, amenities and demand seasonality rather than borough or building age.
- Drop `service fee` from any predictive model to avoid leakage.
- Coach single-listing hosts on calendar management — availability is the clearest professional-host differentiator.
- Position verification as a trust/safety signal, not a ratings lever.
- Focus growth investment on under-supplied boroughs (Bronx, Staten Island) where prices already match Manhattan.

**Limitations:** the dataset lacks booking dates and amenity fields, so peak-season demand, lead times and amenity preferences cannot be measured directly and would need supplementary data.